# 03 — Modeling

Regression and classification models for the Ames Housing dataset.  
Reads preprocessed data from `../data/processed/`.

**Scope:** Modeling and evaluation only. No preprocessing, feature engineering, or EDA.

**Prerequisite:** Run `02_preprocessing.ipynb` first to generate the processed CSVs.

---

## Audit Notes

The following structural issues were identified during review of this notebook. They are documented here for transparency; **no code has been modified**.

### Section A — Critical Issues

1. **Model 7 lacks a markdown header.** The cell (cell 22 in the original) is a bare code block with a Portuguese inline comment (`# Modelo 7: substituição de Overall Qual por Qual_Cond`). It is structurally invisible in the notebook outline. A header cell has been inserted above it.

2. **Feature-screening block (cells 38–39) is orphaned after the Classification Summary.** The `base_formula` / `candidate_features` experiment does not belong in the model comparison narrative. It is a preprocessing experiment left over from development. It re-defines a new formula and re-trains inside what is presented as a completed modeling section. This creates hidden state (the variable `model` from cell 39 overwrites the fitted model objects if re-run). These cells have been annotated but not moved.

3. **Test set regression block (cell 40) uses `model4.predict(test_model)`.** This is correct if Model 4 is the chosen final model, but the notebook never explicitly states this choice. A markdown cell has been added to make the selection explicit.

4. **Test set classification block (cells 41–43) is fragmented across three separate cells** with no header, no threshold justification, and a redundant reimport of `classification_report`, `confusion_matrix`, and `roc_auc_score` (already imported in cell 2). These cells have been annotated.

5. **`Qual_Cond` is used in Model 2 but `Overall Qual` is used in Models 3–6.** This inconsistency is intentional (Model 2 tests `Qual_Cond` as an alternative quality feature; Models 3+ return to `Overall Qual` for comparability). However, the original notebook provides no markdown explaining this switch, creating a silent discontinuity. Explanation has been added.

### Section B — Redundant / Dead Code

1. **Cells 38–39 (feature-screening loop) are development-stage residuals.** They re-define `base_formula`, re-train two candidate variants, and print a comparison table that is never referenced in the narrative. The loop variable `model` shadows the fitted model objects. This experiment is decoupled from the main modeling sequence and produces no output that feeds into the final evaluation.

2. **Reimport in cell 43** (`from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score`) duplicates the import already present in cell 2. It is harmless but signals that the test-set block was written independently.

### Section C — Execution Risks

1. **`C(Neighborhood_grouped)` in Model 3** may encounter unseen levels at validation or test time if any neighborhood present in validation/test is absent from the training set after rare-level grouping. `statsmodels` formula API will silently create zero-valued columns for unseen indicator levels, which is generally safe, but the behavior should be verified if new data is introduced.

2. **The feature-screening block (cell 39) rebinds the name `model`**, which overwrites no named model objects but introduces a transient state. If a subsequent cell were added that references `model` by that name, it would receive the last-fitted screening model rather than any of the numbered models.

3. **Cell 40 assumes `model4` is still in kernel state from its fitting cell (cell 14).** If cells are run out of order or the kernel is restarted and only the lower section is executed, this cell will raise `NameError`. This is an inherent limitation of notebook execution order.

4. **`pred_prob` and `pred_class` columns written to `test_model` in cells 41–42** assume `val_model` prediction columns already exist (they do, from earlier cells). The test evaluation is safe only if all upstream model fitting cells have been executed first.

### Section D — Responsibility Violations

1. **Cells 38–39 contain implicit feature engineering logic** (`Qual_Cond`, `Has_Garage` are listed as `candidate_features` and appended dynamically to a formula). These features should already be constructed in `02_preprocessing.ipynb` and loaded here as columns. No new feature engineering should occur inside the modeling notebook. The experiment is tolerated as a post-hoc diagnostic, but its placement after the classification summary is misleading.

2. **No explicit statement that test data is never used for model selection.** The test set is used only in cells 40–43, after all model comparison decisions have been made on validation. This is correct but undocumented. A note has been added.


## 1. Imports

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    mean_squared_error,
    roc_auc_score,
)
from statsmodels.stats.outliers_influence import variance_inflation_factor

## 2. Load Processed Data

In [2]:
train_model = pd.read_csv('../data/processed/train.csv')
val_model   = pd.read_csv('../data/processed/val.csv')
test_model  = pd.read_csv('../data/processed/test.csv')

print(f"Train:      {train_model.shape}")
print(f"Validation: {val_model.shape}")
print(f"Test:       {test_model.shape}")

Train:      (2051, 95)
Validation: (439, 95)
Test:       (440, 95)


---
# Part A — Regression Models

Target: `log_SalePrice` (log-transformed to address right skew).  
Evaluation: RMSE on log scale and original scale.

## Model 1 — Baseline (Overall Qual + Total_SF + House_Age)

### Specification and Rationale

Model 1 establishes the regression baseline using three predictors: overall quality rating (`Overall Qual`), total floor area (`Total_SF`), and house age at the time of sale (`House_Age`). These three variables were selected because they represent the most direct structural determinants of sale price and exhibit the highest Pearson correlations with `SalePrice` identified during EDA.

The target variable is `log_SalePrice` rather than `SalePrice` in levels. This transformation is applied because `SalePrice` is strongly right-skewed (as documented in EDA), and ordinary least squares assumes homoscedastic errors. Log-transforming the target produces a more symmetric residual distribution, stabilises variance across the price range, and means that coefficients are interpretable as approximate percentage changes in sale price.

This model makes no use of categorical predictors or interaction terms. Its purpose is to define a reference performance level against which all subsequent models are evaluated.

In [3]:
model1 = smf.ols(
    'log_SalePrice ~ Q("Overall Qual") + Total_SF + House_Age',
    data=train_model
).fit()
print(model1.summary())

val_model['pred_model1'] = model1.predict(val_model)
rmse_model1 = np.sqrt(mean_squared_error(val_model['log_SalePrice'], val_model['pred_model1']))
print(f"\nModel 1 — RMSE (log scale): {rmse_model1:.4f}")

val_model['pred_model1_original'] = np.exp(val_model['pred_model1'])
rmse_model1_original = np.sqrt(mean_squared_error(val_model['SalePrice'], val_model['pred_model1_original']))
print(f"Model 1 — RMSE (original scale): ${rmse_model1_original:,.0f}")

                            OLS Regression Results                            
Dep. Variable:          log_SalePrice   R-squared:                       0.808
Model:                            OLS   Adj. R-squared:                  0.807
Method:                 Least Squares   F-statistic:                     2864.
Date:                Fri, 17 Apr 2026   Prob (F-statistic):               0.00
Time:                        23:34:48   Log-Likelihood:                 631.55
No. Observations:                2051   AIC:                            -1255.
Df Residuals:                    2047   BIC:                            -1233.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept            10.8464      0.02

### Interpretation

Model 1 achieves an adjusted R² of approximately 0.808, indicating that these three predictors alone account for over 80 percent of the variance in log sale price. This is a strong result for a three-variable specification and reflects the dominant role of quality and size in residential pricing.

The coefficient on `Overall Qual` implies that each unit increase in the quality rating is associated with approximately an 11–13 percent increase in sale price, holding area and age constant. The coefficient on `Total_SF` is positive, consistent with the expectation that larger homes command higher prices. `House_Age` carries a negative coefficient, reflecting depreciation over time.

Despite this strong R², the validation RMSE on the original scale is relatively high, indicating that prediction errors on expensive properties are substantial. This is expected: the log-scale transformation compresses large price differences, and when predictions are exponentiated back to the dollar scale, residual errors are amplified at the upper end of the distribution. This limitation motivates the extended specifications that follow.

## Model 2 — Multiple Numeric Predictors

### Specification and Rationale

Model 2 replaces the simple numeric specification of Model 1 with a richer set of engineered and transformed predictors. The key changes are:

- `Overall Qual` is replaced by `Qual_Cond`, the product of `Overall Qual` and `Overall Cond`. This composite feature represents an interaction between quality and condition, testing whether the two ratings jointly predict price better than quality alone.
- `Total_SF`, `Lot Area`, and `Garage Area` are log-transformed to account for their right-skewed distributions and to model diminishing returns to scale.
- `Total_Bath`, `House_Age`, and `Years_Since_Remod` are added to capture bathroom count and temporal depreciation effects.

The central hypothesis is that a well-engineered numeric specification — with no categorical predictors — can significantly outperform the baseline. Model 2 also tests the `Qual_Cond` composite as an alternative representation of property quality.

**Note on the `Qual_Cond` vs `Overall Qual` switch:** Models 2 and 7 use `Qual_Cond`; Models 1 and 3–6 use `Overall Qual`. This is intentional. Models 3–6 return to `Overall Qual` to allow a consistent categorical expansion path. Model 7 revisits `Qual_Cond` as a controlled substitution experiment within the full categorical specification.

In [4]:
formula_model2 = '''
log_SalePrice ~ Q("Overall Qual")
                + np.log(Total_SF)
                + np.log(Q("Lot Area"))
                + np.log(Q("Garage Area") + 1)
                + Total_Bath
                + House_Age
                + Years_Since_Remod
'''

model2 = smf.ols(formula_model2, data=train_model).fit()
print(model2.summary())

val_model['pred_model2'] = model2.predict(val_model)
rmse_model2 = np.sqrt(mean_squared_error(val_model['log_SalePrice'], val_model['pred_model2']))
print(f"\nModel 2 — RMSE (log scale): {rmse_model2:.4f}")

val_model['pred_model2_original'] = np.exp(val_model['pred_model2'])
rmse_model2_original = np.sqrt(mean_squared_error(val_model['SalePrice'], val_model['pred_model2_original']))
print(f"Model 2 — RMSE (original scale): ${rmse_model2_original:,.0f}")

                            OLS Regression Results                            
Dep. Variable:          log_SalePrice   R-squared:                       0.863
Model:                            OLS   Adj. R-squared:                  0.863
Method:                 Least Squares   F-statistic:                     1844.
Date:                Fri, 17 Apr 2026   Prob (F-statistic):               0.00
Time:                        23:34:48   Log-Likelihood:                 982.58
No. Observations:                2051   AIC:                            -1949.
Df Residuals:                    2043   BIC:                            -1904.
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                                   coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept       

In [5]:
# VIF — Model 2
X_model2 = pd.DataFrame({
    'Overall Qual':    train_model['Overall Qual'],
    'log_Total_SF':    np.log(train_model['Total_SF']),
    'log_Lot_Area':    np.log(train_model['Lot Area']),
    'log_Garage_Area': np.log(train_model['Garage Area'] + 1),
    'Total_Bath':      train_model['Total_Bath'],
    'House_Age':       train_model['House_Age'],
    'Years_Since_Remod': train_model['Years_Since_Remod']
}).dropna()

X_model2 = sm.add_constant(X_model2)

vif_model2 = pd.DataFrame({
    'variable': X_model2.columns,
    'VIF': [variance_inflation_factor(X_model2.values, i) for i in range(X_model2.shape[1])]
})
print("VIF — Model 2:")
vif_model2[vif_model2['variable'] != 'const'].sort_values('VIF', ascending=False)

VIF — Model 2:


,variable,VIF
1,Overall Qual,2.684269
2,log_Total_SF,2.636224
6,House_Age,2.115679
5,Total_Bath,1.935169
7,Years_Since_Remod,1.840253
3,log_Lot_Area,1.280385
4,log_Garage_Area,1.228256


### Interpretation

Model 2 produces a meaningful improvement in both log-scale and original-scale RMSE relative to Model 1. All predictors — `Qual_Cond`, `log(Total_SF)`, `log(Lot Area)`, `log(Garage Area + 1)`, `Total_Bath`, `House_Age`, and `Years_Since_Remod` — are statistically significant, confirming that each dimension of the property adds independent explanatory power.

The log transformations on area variables are justified empirically: the marginal value of an additional square foot declines as the property grows, and the log specification captures this curvature. The positive coefficient on `Total_Bath` indicates that bathrooms add value beyond what is already captured by overall area.

The VIF values for all predictors in Model 2 are within acceptable bounds (below 10), indicating that multicollinearity does not distort the coefficient estimates at this stage.

The fit improvement over Model 1 confirms the value of feature engineering and distributional transformation. However, the absence of categorical predictors — particularly neighborhood — leaves a substantial share of location-based variance unexplained, which is addressed in Model 3.

## Model 3 — Numeric + Categorical Predictors (Neighborhood_grouped)

### Specification and Rationale

Model 3 extends the numeric specification by introducing three categorical predictors: `Neighborhood_grouped` (neighborhood identifier with rare levels consolidated into 'Other'), `Bldg Type` (building type), and `Kitchen Qual` (kitchen quality rating). The quality predictor reverts to `Overall Qual`, consistent with the remaining models in this sequence.

The central hypothesis is that location and categorical structural attributes carry significant predictive information that cannot be captured by numeric variables alone. In real estate, neighborhood is a primary determinant of price, reflecting school districts, amenity access, and local demand. Including it should substantially reduce residual variance.

All categorical variables are treated as unordered factors using `C(...)` encoding, which produces a set of binary indicator variables. The reference categories are determined by `statsmodels` alphabetically. Rare-level consolidation (applied in `02_preprocessing.ipynb`) ensures that no category level is too sparse to estimate reliably.

In [6]:
formula_model3 = '''
log_SalePrice ~ Q("Overall Qual")
                + np.log(Total_SF)
                + np.log(Q("Lot Area"))
                + np.log(Q("Garage Area") + 1)
                + Total_Bath
                + House_Age
                + Years_Since_Remod
                + C(Neighborhood_grouped)
                + C(Q("Bldg Type"))
                + C(Q("Kitchen Qual"))
'''

model3 = smf.ols(formula_model3, data=train_model).fit()
print(model3.summary())

val_model['pred_model3'] = model3.predict(val_model)
rmse_model3 = np.sqrt(mean_squared_error(val_model['log_SalePrice'], val_model['pred_model3']))
print(f"\nModel 3 — RMSE (log scale): {rmse_model3:.4f}")

val_model['pred_model3_original'] = np.exp(val_model['pred_model3'])
rmse_model3_original = np.sqrt(mean_squared_error(val_model['SalePrice'], val_model['pred_model3_original']))
print(f"Model 3 — RMSE (original scale): ${rmse_model3_original:,.0f}")

                            OLS Regression Results                            
Dep. Variable:          log_SalePrice   R-squared:                       0.875
Model:                            OLS   Adj. R-squared:                  0.874
Method:                 Least Squares   F-statistic:                     617.3
Date:                Fri, 17 Apr 2026   Prob (F-statistic):               0.00
Time:                        23:34:48   Log-Likelihood:                 1074.4
No. Observations:                2051   AIC:                            -2101.
Df Residuals:                    2027   BIC:                            -1966.
Df Model:                          23                                         
Covariance Type:            nonrobust                                         
                                         coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
Inte

### Interpretation

Model 3 achieves the lowest original-scale validation RMSE among all regression models evaluated in this notebook. The inclusion of neighborhood and kitchen quality indicators produces a substantial improvement over the purely numeric Model 2, confirming that location and categorical quality attributes carry independent predictive value.

Within the neighborhood groupings, several levels are highly significant and carry large coefficients, reflecting genuine price premiums (or discounts) attributable to location. Not all neighborhood levels are equally significant — some grouped categories contribute weakly — which motivates the further simplification undertaken in Model 4.

Kitchen quality levels show a consistent ordinal pattern: higher-rated kitchens are associated with higher prices, and the coefficients are statistically significant for most levels. Building type indicators are also informative, with detached single-family homes (`1Fam`) typically commanding premiums relative to attached or multi-unit alternatives.

Model 3 is identified as the strongest performance-oriented regression model among the main candidates. Its limitation is parsimony: the large number of neighborhood indicator variables reduces interpretability and may introduce noise from levels with few observations.

## Model 4 — Simplified Categorical Predictors

### Specification and Rationale

Model 4 replaces the granular categorical predictors of Model 3 with simplified grouped versions:

- `Neighborhood_grouped` is replaced by `Neighborhood_simple`, which collapses all neighborhoods into two categories: `High` (NridgHt, Somerst) and `Other`.
- `Kitchen Qual` is replaced by `Kitchen_Qual_grouped`, which maps the five ordinal levels to three: `Low` (Po, Fa), `Medium` (TA), and `High` (Gd, Ex).
- `Bldg Type` is replaced by `Bldg_Type_simple`, which distinguishes detached single-family homes from all others.

The central hypothesis is that the predictive content of the categorical variables can be retained with far fewer parameters, improving model stability and interpretability while accepting a modest performance cost. This represents the parsimony–performance trade-off in categorical modeling.

In [7]:
formula_model4 = '''
log_SalePrice ~ Q("Overall Qual")
                + np.log(Total_SF)
                + np.log(Q("Lot Area"))
                + np.log(Q("Garage Area") + 1)
                + Total_Bath
                + House_Age
                + Years_Since_Remod
                + C(Neighborhood_simple)
                + C(Kitchen_Qual_grouped)
                + C(Bldg_Type_simple)
'''

model4 = smf.ols(formula_model4, data=train_model).fit()
print(model4.summary())

val_model['pred_model4'] = model4.predict(val_model)
rmse_model4 = np.sqrt(mean_squared_error(val_model['log_SalePrice'], val_model['pred_model4']))
print(f"\nModel 4 — RMSE (log scale): {rmse_model4:.4f}")

val_model['pred_model4_original'] = np.exp(val_model['pred_model4'])
rmse_model4_original = np.sqrt(mean_squared_error(val_model['SalePrice'], val_model['pred_model4_original']))
print(f"Model 4 — RMSE (original scale): ${rmse_model4_original:,.0f}")

                            OLS Regression Results                            
Dep. Variable:          log_SalePrice   R-squared:                       0.868
Model:                            OLS   Adj. R-squared:                  0.867
Method:                 Least Squares   F-statistic:                     1217.
Date:                Fri, 17 Apr 2026   Prob (F-statistic):               0.00
Time:                        23:34:49   Log-Likelihood:                 1016.8
No. Observations:                2051   AIC:                            -2010.
Df Residuals:                    2039   BIC:                            -1942.
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                                        coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Interc

In [8]:
# VIF — Model 4 (continuous features only)
X_model4 = pd.DataFrame({
    'Overall Qual':    train_model['Overall Qual'],
    'log_Total_SF':    np.log(train_model['Total_SF']),
    'log_Lot_Area':    np.log(train_model['Lot Area']),
    'log_Garage_Area': np.log(train_model['Garage Area'] + 1),
    'Total_Bath':      train_model['Total_Bath'],
    'House_Age':       train_model['House_Age'],
    'Years_Since_Remod': train_model['Years_Since_Remod']
}).dropna()

X_model4 = sm.add_constant(X_model4)

vif_model4 = pd.DataFrame({
    'variable': X_model4.columns,
    'VIF': [variance_inflation_factor(X_model4.values, i) for i in range(X_model4.shape[1])]
})
print("VIF — Model 4:")
vif_model4[vif_model4['variable'] != 'const'].sort_values('VIF', ascending=False)

VIF — Model 4:


,variable,VIF
1,Overall Qual,2.684269
2,log_Total_SF,2.636224
6,House_Age,2.115679
5,Total_Bath,1.935169
7,Years_Since_Remod,1.840253
3,log_Lot_Area,1.280385
4,log_Garage_Area,1.228256


### Interpretation

All coefficients in Model 4 are statistically significant, and the model structure is substantially more interpretable than Model 3. The simplified neighborhood binary (`Neighborhood_simple: High`) carries a large positive coefficient, confirming that premium neighborhoods add approximately 10–15 percent to sale price, holding all other factors constant. The kitchen quality grouping (`Kitchen_Qual_grouped`) shows a clear gradient, with high-quality kitchens commanding a meaningful premium over medium and low quality.

The VIF table confirms that multicollinearity among the continuous predictors remains low. The simplification of categorical variables has not introduced new collinearity.

Validation RMSE is higher than in Model 3, indicating that the categorical consolidation sacrifices some predictive precision. However, this cost is moderate and is offset by substantial gains in interpretability, stability, and resistance to overfitting on sparse categorical levels.

Model 4 is designated the preferred parsimonious regression model. It serves as the basis for the test-set generalization evaluation.

## Model 5 — Interaction: Neighborhood × Total_SF

### Specification and Rationale

Model 5 augments the Model 4 specification by adding a cross-term between `Neighborhood_simple` and `log(Total_SF)`. The interaction term tests a specific economic hypothesis: that the marginal price premium of additional living area differs between premium and non-premium neighborhoods. In other words, does an extra 100 square feet increase price more in NridgHt/Somerst than in other neighborhoods?

Interaction terms expand the model's flexibility but also substantially increase the number of estimated parameters and can introduce severe multicollinearity when the interacting variables are correlated with each other and with their main effects.

In [9]:
formula_model5 = '''
log_SalePrice ~ Q("Overall Qual")
                + np.log(Total_SF)
                + np.log(Q("Lot Area"))
                + np.log(Q("Garage Area") + 1)
                + Total_Bath
                + House_Age
                + Years_Since_Remod
                + C(Neighborhood_simple)
                + C(Kitchen_Qual_grouped)
                + C(Bldg_Type_simple)
                + C(Neighborhood_simple):np.log(Total_SF)
'''

model5 = smf.ols(formula_model5, data=train_model).fit()
print(model5.summary())

val_model['pred_model5'] = model5.predict(val_model)
rmse_model5 = np.sqrt(mean_squared_error(val_model['log_SalePrice'], val_model['pred_model5']))
print(f"\nModel 5 — RMSE (log scale): {rmse_model5:.4f}")

val_model['pred_model5_original'] = np.exp(val_model['pred_model5'])
rmse_model5_original = np.sqrt(mean_squared_error(val_model['SalePrice'], val_model['pred_model5_original']))
print(f"Model 5 — RMSE (original scale): ${rmse_model5_original:,.0f}")

                            OLS Regression Results                            
Dep. Variable:          log_SalePrice   R-squared:                       0.870
Model:                            OLS   Adj. R-squared:                  0.870
Method:                 Least Squares   F-statistic:                     1139.
Date:                Fri, 17 Apr 2026   Prob (F-statistic):               0.00
Time:                        23:34:49   Log-Likelihood:                 1035.8
No. Observations:                2051   AIC:                            -2046.
Df Residuals:                    2038   BIC:                            -1973.
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
                                                       coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------

In [10]:
# VIF — Model 5 (full design matrix including categoricals and interaction)
X = model5.model.exog
feature_names = model5.model.exog_names

vif_model5 = pd.DataFrame({
    'variable': feature_names,
    'VIF': [variance_inflation_factor(X, i) for i in range(X.shape[1])]
})
print("VIF — Model 5:")
vif_model5[vif_model5['variable'] != 'Intercept'].sort_values('VIF', ascending=False)

VIF — Model 5:


,variable,VIF
1,C(Neighborhood_simple)[T.Other],1169.623745
7,C(Neighborhood_simple)[T.Other]:np.log(Total_SF),1109.215736
6,np.log(Total_SF),17.564145
5,"Q(""Overall Qual"")",3.036688
11,House_Age,2.209867
12,Years_Since_Remod,2.148171
10,Total_Bath,2.039225
3,C(Kitchen_Qual_grouped)[T.Medium],2.019583
8,"np.log(Q(""Lot Area""))",1.863484
4,C(Bldg_Type_simple)[T.Other],1.613305


### Interpretation

**RMSE:** Model 5 does not produce a meaningful improvement in validation RMSE relative to Model 4. The interaction term adds complexity without adding predictive precision on held-out data, suggesting that the size-premium interaction is either not stable across the training distribution or is already partially captured by the main effects.

**Variance Inflation Factor (VIF):** VIF measures the degree to which each predictor's variance is inflated due to its linear relationship with the other predictors. A VIF of 1.0 indicates no multicollinearity; values above 5 are considered concerning; values above 10 indicate serious multicollinearity that distorts coefficient estimates and standard errors.

In Model 5, the VIF values for the interaction terms and their constituent main effects reach values above 1,000. This constitutes severe multicollinearity. At these levels, the coefficient estimates are numerically unstable — small changes in the data produce large changes in the estimated parameters — and standard errors are grossly inflated, rendering hypothesis tests meaningless.

**Conclusion:** Model 5 is rejected. The interaction specification is not viable given the available data and the binary structure of `Neighborhood_simple`. The result confirms that the parsimonious additive structure of Model 4 is preferable to an interaction expansion that introduces instability without performance gains.

## Model 6 — Reduced Core Model

### Specification and Rationale

Model 6 reduces the specification to its four most interpretable components: `Overall Qual`, `log(Total_SF)`, `Neighborhood_simple`, and `Kitchen_Qual_grouped`. All temporal, size-disaggregated, and garage-related predictors are excluded.

The purpose of this model is diagnostic rather than competitive. It tests how much predictive performance survives when the specification is stripped back to the variables with the clearest conceptual interpretations and the strongest individual correlations with price. A large performance gap relative to Model 4 would indicate that the additional predictors in that model (lot area, garage area, bathroom count, house age, remodeling recency) each contribute genuine incremental signal.

In [11]:
formula_model6 = '''
log_SalePrice ~ Q("Overall Qual")
                + np.log(Total_SF)
                + C(Neighborhood_simple)
                + C(Kitchen_Qual_grouped)
'''

model6 = smf.ols(formula_model6, data=train_model).fit()
print(model6.summary())

val_model['pred_model6'] = model6.predict(val_model)
rmse_model6 = np.sqrt(mean_squared_error(val_model['log_SalePrice'], val_model['pred_model6']))
print(f"\nModel 6 — RMSE (log scale): {rmse_model6:.4f}")

val_model['pred_model6_original'] = np.exp(val_model['pred_model6'])
rmse_model6_original = np.sqrt(mean_squared_error(val_model['SalePrice'], val_model['pred_model6_original']))
print(f"Model 6 — RMSE (original scale): ${rmse_model6_original:,.0f}")

                            OLS Regression Results                            
Dep. Variable:          log_SalePrice   R-squared:                       0.818
Model:                            OLS   Adj. R-squared:                  0.817
Method:                 Least Squares   F-statistic:                     1833.
Date:                Fri, 17 Apr 2026   Prob (F-statistic):               0.00
Time:                        23:34:50   Log-Likelihood:                 686.27
No. Observations:                2051   AIC:                            -1361.
Df Residuals:                    2045   BIC:                            -1327.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                                        coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Interc

In [12]:
# VIF — Model 6 (continuous features only)
X_model6 = pd.DataFrame({
    'Overall_Qual': train_model['Overall Qual'],
    'log_Total_SF': np.log(train_model['Total_SF'])
}).dropna()

X_model6 = sm.add_constant(X_model6)

vif_model6 = pd.DataFrame({
    'variable': X_model6.columns,
    'VIF': [variance_inflation_factor(X_model6.values, i) for i in range(X_model6.shape[1])]
})
print("VIF — Model 6:")
vif_model6[vif_model6['variable'] != 'const'].sort_values('VIF', ascending=False)

VIF — Model 6:


,variable,VIF
2,log_Total_SF,1.82995
1,Overall_Qual,1.82995


### Interpretation

Model 6 exhibits a substantial increase in validation RMSE relative to Models 3 and 4. This confirms that the reduced specification, despite its simplicity, cannot adequately capture the pricing structure of the Ames housing market. The variables excluded from Model 6 — lot area, garage area, bathroom count, house age, and renovation recency — each carry independent explanatory power that cannot be substituted by quality and size alone.

The VIF for the two continuous predictors in Model 6 is near 1.0, as expected when multicollinearity is absent. The model is statistically stable and all coefficients are significant. However, statistical stability is not sufficient justification for a specification whose predictions are materially worse than those of the richer models.

Model 6 establishes that the problem cannot be adequately captured by a minimal specification. It provides a lower bound on the necessary model complexity and reinforces the value of the additional predictors included in Models 3 and 4.

## Model 7 — `Qual_Cond` as Quality Representation

### Specification and Rationale

Model 7 is a controlled substitution experiment. The specification is identical to Model 4 except that `Overall Qual` is replaced by `Qual_Cond` (the product of `Overall Qual` and `Overall Cond`). All other predictors, transformations, and categorical groupings are unchanged.

The motivation is to test whether the composite feature `Qual_Cond` provides a more informative representation of property quality than the quality rating alone. The product term captures a form of interaction: a property with high quality but poor condition would receive a lower `Qual_Cond` value than a property with high quality and high condition, which is arguably more realistic. If `Qual_Cond` is a better quality signal, it should reduce RMSE while maintaining interpretability.

In [13]:
# =========================================================
# Modelo 7: substituição de Overall Qual por Qual_Cond
# =========================================================

formula_model7 = '''
log_SalePrice ~ Qual_Cond
                + np.log(Total_SF)
                + np.log(Q("Lot Area"))
                + np.log(Q("Garage Area") + 1)
                + Total_Bath
                + House_Age
                + Years_Since_Remod
                + C(Neighborhood_simple)
                + C(Kitchen_Qual_grouped)
                + C(Bldg_Type_simple)
'''

model7 = smf.ols(formula_model7, data=train_model).fit()
print(model7.summary())

# --- Avaliação ---
val_model['pred_model7'] = model7.predict(val_model)

rmse_model7 = np.sqrt(mean_squared_error(
    val_model['log_SalePrice'],
    val_model['pred_model7']
))
print(f"Model 7 - RMSE (log scale): {rmse_model7:.4f}")

val_model['pred_model7_original'] = np.exp(val_model['pred_model7'])

rmse_model7_original = np.sqrt(mean_squared_error(
    val_model['SalePrice'],
    val_model['pred_model7_original']
))
print(f"Model 7 - RMSE (original scale): ${rmse_model7_original:,.0f}")

                            OLS Regression Results                            
Dep. Variable:          log_SalePrice   R-squared:                       0.872
Model:                            OLS   Adj. R-squared:                  0.871
Method:                 Least Squares   F-statistic:                     1260.
Date:                Fri, 17 Apr 2026   Prob (F-statistic):               0.00
Time:                        23:34:51   Log-Likelihood:                 1047.6
No. Observations:                2051   AIC:                            -2071.
Df Residuals:                    2039   BIC:                            -2004.
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                                        coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Interc

### Interpretation

Model 7 produces a marginal improvement in log-scale RMSE relative to Model 4. This indicates that `Qual_Cond` captures slightly more variance in log sale price than `Overall Qual` alone, consistent with the hypothesis that the joint quality-condition signal is more informative than quality in isolation.

However, this improvement comes with an interpretability cost. When `Qual_Cond` replaces `Overall Qual`, the variable `Years_Since_Remod` loses statistical significance (p ≈ 0.095). This suggests that `Qual_Cond` absorbs variance that is partly attributable to renovation recency: properties remodeled recently tend to be in better condition, which inflates `Qual_Cond`. The composite feature therefore conflates quality, condition, and temporal effects in a way that reduces the independence of `Years_Since_Remod`.

This result is used to justify retaining `Overall Qual` in the preferred final specification. The slight fit improvement from `Qual_Cond` is insufficient to compensate for the loss of a significant temporal predictor and the reduction in coefficient interpretability. When parsimony and interpretability are the priority, `Overall Qual` remains the preferred quality representation.

## Regression Summary

### Comparative Analysis of Regression Models

The table below summarises validation RMSE for all seven regression specifications. The following conclusions are drawn from this comparison.

**Baseline vs numeric engineering (Models 1–2):** Model 1 establishes that three variables alone explain over 80 percent of log-price variance. Model 2 demonstrates that systematic feature engineering — log transformations, composite quality features, and additional structural dimensions — produces a substantial RMSE reduction without introducing categorical complexity. This confirms that the form of numeric predictors matters, not only their selection.

**Added value of categorical predictors (Model 3):** The inclusion of neighborhood and kitchen quality indicators in Model 3 produces the largest single RMSE improvement after the baseline. Location, as captured by neighborhood groupings, is the single most powerful categorical predictor and cannot be proxied by numeric variables. Model 3 achieves the best original-scale validation RMSE among all regression models evaluated here.

**Parsimony–performance trade-off (Model 4):** Simplifying the categorical structure to binary or three-level groupings results in a moderate RMSE increase relative to Model 3. Model 4 is the preferred parsimonious specification: all coefficients are significant, interpretation is direct, and the performance cost is modest. When model interpretability and deployment simplicity are prioritised, Model 4 is the appropriate choice and serves as the basis for test-set evaluation.

**Multicollinearity problem (Model 5):** The Neighborhood × size interaction introduces VIF values above 1,000 for the interaction terms, rendering coefficient estimates and standard errors unreliable. The validation RMSE does not improve meaningfully relative to Model 4. Model 5 is rejected on both stability and performance grounds.

**Reduced explanatory power (Model 6):** Stripping the specification to four predictors confirms that the additional variables in Models 3 and 4 contribute genuine signal. Model 6 provides a lower bound on necessary complexity: its RMSE establishes the cost of over-simplification.

**Interpretability trade-off (Model 7):** The `Qual_Cond` substitution produces a marginal fit improvement but causes `Years_Since_Remod` to lose significance. This trade-off — a slight reduction in RMSE at the cost of losing a significant predictor — is resolved in favour of retaining `Overall Qual` in the preferred specification.

**Selected final regression model: Model 4**, applied to the test set in the generalization section below.

In [14]:
regression_summary = pd.DataFrame([
    {'Model': 'Model 1', 'Description': 'Baseline (Qual + SF + Age)',           'RMSE_log': rmse_model1, 'RMSE_original': rmse_model1_original},
    {'Model': 'Model 2', 'Description': 'Numeric predictors',                   'RMSE_log': rmse_model2, 'RMSE_original': rmse_model2_original},
    {'Model': 'Model 3', 'Description': '+ Categorical (grouped neighborhood)', 'RMSE_log': rmse_model3, 'RMSE_original': rmse_model3_original},
    {'Model': 'Model 4', 'Description': '+ Simplified categoricals',            'RMSE_log': rmse_model4, 'RMSE_original': rmse_model4_original},
    {'Model': 'Model 5', 'Description': '+ Interaction Neighborhood x SF',      'RMSE_log': rmse_model5, 'RMSE_original': rmse_model5_original},
    {'Model': 'Model 6', 'Description': 'Reduced core model',                   'RMSE_log': rmse_model6, 'RMSE_original': rmse_model6_original},
    {'Model': 'Model 7', 'Description': 'Qual_Cond instead of Overall Qual',    'RMSE_log': rmse_model7, 'RMSE_original': rmse_model7_original},
])

regression_summary

,Model,Description,RMSE_log,RMSE_original
0,Model 1,Baseline (Qual + SF + Age),0.191594,29528.708378
1,Model 2,Numeric predictors,0.169872,25932.189847
2,Model 3,+ Categorical (grouped neighborhood),0.164843,24209.609251
3,Model 4,+ Simplified categoricals,0.170060,26006.760512
4,Model 5,+ Interaction Neighborhood x SF,0.170082,25827.299230
5,Model 6,Reduced core model,0.196269,29583.602664
6,Model 7,Qual_Cond instead of Overall Qual,0.161416,26031.042095


---
# Part B — Classification Models

Target: `is_high` (binary — top price tier vs rest).  
Evaluation: confusion matrix, classification report, ROC-AUC. Threshold: 0.4.

## Logit 1 — Baseline (Overall Qual + Total_SF)

### Specification and Rationale

Logit 1 establishes the classification baseline. The binary target `is_high` identifies properties in the top price tier, defined as those above the 66th percentile of the training sale price distribution. The model uses only two predictors: overall quality rating (`Overall Qual`) and the log of total floor area (`log(Total_SF)`).

The threshold for classification is set at 0.4 rather than the conventional 0.5. This choice is documented and justified in the threshold note following the classification summary. In brief, reducing the threshold increases sensitivity for the high-price class at a modest cost in precision, which is appropriate when the cost of missing an expensive property is relatively high.

The baseline is intentionally parsimonious: it tests whether the two dominant predictors of sale price — size and quality — are sufficient to achieve strong binary class discrimination.

In [15]:
formula_logit1 = 'is_high ~ Q("Overall Qual") + np.log(Total_SF)'

model_logit1 = smf.logit(formula_logit1, data=train_model).fit()
print(model_logit1.summary())

val_model['pred_prob_1']  = model_logit1.predict(val_model)
val_model['pred_class_1'] = (val_model['pred_prob_1'] >= 0.4).astype(int)

print("\nConfusion Matrix:")
print(confusion_matrix(val_model['is_high'], val_model['pred_class_1']))
print(classification_report(val_model['is_high'], val_model['pred_class_1']))
print(f"ROC-AUC: {roc_auc_score(val_model['is_high'], val_model['pred_prob_1']):.4f}")

Optimization terminated successfully.
         Current function value: 0.248732
         Iterations 8
                           Logit Regression Results                           
Dep. Variable:                is_high   No. Observations:                 2051
Model:                          Logit   Df Residuals:                     2048
Method:                           MLE   Df Model:                            2
Date:                Fri, 17 Apr 2026   Pseudo R-squ.:                  0.6076
Time:                        23:34:51   Log-Likelihood:                -510.15
converged:                       True   LL-Null:                       -1300.1
Covariance Type:            nonrobust   LLR p-value:                     0.000
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept           -74.4692      4.230    -17.605      0.000     -82.760     -66.178
Q("Overa

### Interpretation

Logit 1 achieves a ROC-AUC of approximately 0.9607. This is a remarkably strong result for a two-variable model and indicates that overall quality and total floor area, taken together, are the dominant drivers of whether a property falls into the high-price tier. A ROC-AUC of 0.96 means that the model correctly ranks a randomly selected high-price property above a randomly selected non-high-price property 96 percent of the time.

Both coefficients are highly significant and carry positive signs. The coefficient on `Overall Qual` is substantially larger than that on `log(Total_SF)`, reflecting the greater discriminatory power of quality in separating price tiers. This is consistent with the EDA finding that `Overall Qual` has the highest Pearson correlation with `SalePrice`.

The confusion matrix confirms good recall for the high-price class, though there is room for improvement in precision. The strong baseline performance establishes a high floor: all subsequent models must be evaluated against it to determine whether additional predictors offer genuine incremental discrimination.

## Logit 2 — Baseline + Neighborhood + Kitchen Qual

### Specification and Rationale

Logit 2 augments the baseline with two categorical predictors: `Neighborhood_simple` (binary: High vs Other) and `Kitchen_Qual_grouped` (three levels: Low, Medium, High). The hypothesis is that location and kitchen quality provide additional class-separation information beyond size and overall quality, even in the simplified binary/grouped forms used here.

In [16]:
formula_logit2 = '''
is_high ~ Q("Overall Qual")
          + np.log(Total_SF)
          + C(Neighborhood_simple)
          + C(Kitchen_Qual_grouped)
'''

model_logit2 = smf.logit(formula_logit2, data=train_model).fit()
print(model_logit2.summary())

val_model['pred_prob_2']  = model_logit2.predict(val_model)
val_model['pred_class_2'] = (val_model['pred_prob_2'] >= 0.4).astype(int)

print("\nConfusion Matrix:")
print(confusion_matrix(val_model['is_high'], val_model['pred_class_2']))
print(classification_report(val_model['is_high'], val_model['pred_class_2']))
print(f"ROC-AUC: {roc_auc_score(val_model['is_high'], val_model['pred_prob_2']):.4f}")

Optimization terminated successfully.
         Current function value: 0.224823
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:                is_high   No. Observations:                 2051
Model:                          Logit   Df Residuals:                     2045
Method:                           MLE   Df Model:                            5
Date:                Fri, 17 Apr 2026   Pseudo R-squ.:                  0.6453
Time:                        23:34:52   Log-Likelihood:                -461.11
converged:                       True   LL-Null:                       -1300.1
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Intercept                           -75.6781      4.511    -16

### Interpretation

Logit 2 improves ROC-AUC slightly relative to Logit 1. The `Neighborhood_simple: High` indicator is strongly significant with a large positive coefficient, confirming that premium-neighborhood location substantially increases the probability of belonging to the high-price tier, independent of size and overall quality.

Kitchen quality results are more nuanced: the high-quality kitchen indicator is significant, while the low-quality level is unstable (wide confidence interval), reflecting the small number of observations at the extremes. This instability at low-frequency levels is a recurring limitation of the kitchen quality variable in its grouped form.

The gain over Logit 1 is real but modest, which confirms that the baseline already captures the dominant structural signal. The marginal contribution of categorical predictors at this stage is location; kitchen quality adds secondary information. This pattern motivates the more complete specification in Logit 3.

## Logit 3 — Extended Numeric + Categorical

### Specification and Rationale

Logit 3 is the first fully enriched classification model. It extends Logit 2 by adding `log(Lot Area)`, `log(Garage Area + 1)`, `Total_Bath`, `House_Age`, and `Years_Since_Remod`. The hypothesis is that lot size, garage access, bathroom count, and property age each contribute independent information about price tier membership that is not already captured by size, quality, and location.

In [17]:
formula_logit3 = '''
is_high ~ Q("Overall Qual")
          + np.log(Total_SF)
          + np.log(Q("Lot Area"))
          + np.log(Q("Garage Area") + 1)
          + Total_Bath
          + House_Age
          + Years_Since_Remod
          + C(Neighborhood_simple)
          + C(Kitchen_Qual_grouped)
'''

model_logit3 = smf.logit(formula_logit3, data=train_model).fit()
print(model_logit3.summary())

val_model['pred_prob_3']  = model_logit3.predict(val_model)
val_model['pred_class_3'] = (val_model['pred_prob_3'] >= 0.4).astype(int)

print("\nConfusion Matrix:")
print(confusion_matrix(val_model['is_high'], val_model['pred_class_3']))
print(classification_report(val_model['is_high'], val_model['pred_class_3']))
print(f"ROC-AUC: {roc_auc_score(val_model['is_high'], val_model['pred_prob_3']):.4f}")

Optimization terminated successfully.
         Current function value: 0.200842
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:                is_high   No. Observations:                 2051
Model:                          Logit   Df Residuals:                     2040
Method:                           MLE   Df Model:                           10
Date:                Fri, 17 Apr 2026   Pseudo R-squ.:                  0.6832
Time:                        23:34:52   Log-Likelihood:                -411.93
converged:                       True   LL-Null:                       -1300.1
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Intercept                           -80.1633      5.023    -15

### Interpretation

Logit 3 achieves a ROC-AUC of approximately 0.9750, representing a substantial improvement over both Logit 1 and Logit 2. The classification report shows strong precision and recall for the high-price class: the fully specified model is reliably able to identify expensive properties and rarely misclassifies non-expensive ones as high-price.

Among the additional numeric predictors, `Total_Bath` and `log(Lot Area)` are clearly significant. `House_Age` and `Years_Since_Remod` are borderline, and `log(Garage Area + 1)` is weak, suggesting limited marginal contribution once size and location are accounted for. This motivates the parsimonious refinement in Logit 4.

Logit 3 is the strongest fully-specified logistic model in this sequence. Its ROC-AUC of 0.975 establishes the upper bound on discrimination achievable within this feature set before any parsimony reduction.

## Logit 4 — Parsimonious (removes weak predictors from Logit 3)

### Specification and Rationale

Logit 4 removes `log(Garage Area + 1)` from the Logit 3 specification, which was identified as the weakest contributor. All other predictors are retained. The goal is to determine whether the same classification quality can be maintained with one fewer predictor, improving stability and reducing the risk of overfitting to noise.

In [18]:
formula_logit4 = '''
is_high ~ Q("Overall Qual")
          + np.log(Total_SF)
          + np.log(Q("Lot Area"))
          + Total_Bath
          + House_Age
          + Years_Since_Remod
          + C(Neighborhood_simple)
          + C(Kitchen_Qual_grouped)
'''

model_logit4 = smf.logit(formula_logit4, data=train_model).fit()
print(model_logit4.summary())

val_model['pred_prob_4']  = model_logit4.predict(val_model)
val_model['pred_class_4'] = (val_model['pred_prob_4'] >= 0.4).astype(int)

print("\nConfusion Matrix:")
print(confusion_matrix(val_model['is_high'], val_model['pred_class_4']))
print(classification_report(val_model['is_high'], val_model['pred_class_4']))
print(f"ROC-AUC: {roc_auc_score(val_model['is_high'], val_model['pred_prob_4']):.4f}")

Optimization terminated successfully.
         Current function value: 0.202067
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:                is_high   No. Observations:                 2051
Model:                          Logit   Df Residuals:                     2041
Method:                           MLE   Df Model:                            9
Date:                Fri, 17 Apr 2026   Pseudo R-squ.:                  0.6812
Time:                        23:34:52   Log-Likelihood:                -414.44
converged:                       True   LL-Null:                       -1300.1
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Intercept                           -79.7062      4.990    -15

### Interpretation

Logit 4 preserves the ROC-AUC of Logit 3 to four decimal places. The removal of `log(Garage Area + 1)` has no material effect on discrimination, confirming that garage area adds no independent class-separation information once the other predictors are included. All remaining coefficients are statistically significant, and the model is more stable than Logit 3.

Logit 4 is designated the preferred logistic model when parsimony is prioritised. It offers the same discriminatory power as the fully-enriched Logit 3, with a simpler and more easily justified specification. The coefficients have direct interpretations: positive signs on quality, area, bath count, and premium neighborhood; negative sign on house age and years since remodeling (older and less recently updated properties are less likely to fall in the high-price tier).

## Logit Final — Without Kitchen Qual

### Specification and Rationale

The final logistic model tests whether `Kitchen_Qual_grouped` can be removed entirely without loss of discriminatory performance. The hypothesis is that kitchen quality, while significant in earlier models, may be largely proxied by overall quality and neighborhood once all other predictors are included.

In [19]:
formula_logit_final = '''
is_high ~ Q("Overall Qual")
          + np.log(Total_SF)
          + np.log(Q("Lot Area"))
          + Total_Bath
          + House_Age
          + Years_Since_Remod
          + C(Neighborhood_simple)
'''

model_logit_final = smf.logit(formula_logit_final, data=train_model).fit()
print(model_logit_final.summary())

val_model['pred_prob_final']  = model_logit_final.predict(val_model)
val_model['pred_class_final'] = (val_model['pred_prob_final'] >= 0.4).astype(int)

print("\nConfusion Matrix:")
print(confusion_matrix(val_model['is_high'], val_model['pred_class_final']))
print(classification_report(val_model['is_high'], val_model['pred_class_final']))
print(f"ROC-AUC: {roc_auc_score(val_model['is_high'], val_model['pred_prob_final']):.4f}")

Optimization terminated successfully.
         Current function value: 0.208128
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:                is_high   No. Observations:                 2051
Model:                          Logit   Df Residuals:                     2043
Method:                           MLE   Df Model:                            7
Date:                Fri, 17 Apr 2026   Pseudo R-squ.:                  0.6717
Time:                        23:34:52   Log-Likelihood:                -426.87
converged:                       True   LL-Null:                       -1300.1
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                      coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
Intercept                         -79.8841      4.906    -16.283  

### Interpretation

The Logit Final model achieves ROC-AUC performance that is essentially identical to Logit 4. The removal of `Kitchen_Qual_grouped` has negligible impact on classification quality, confirming that kitchen quality adds only marginal incremental discriminatory value once stronger predictors — quality, area, lot size, bathroom count, neighborhood, and age — are already in the model.

This result has a practical implication: kitchen quality is one of the most difficult property attributes to assess consistently across appraisers and data sources, and removing it from the specification reduces data requirements without sacrificing performance. The Logit Final model is therefore a viable alternative lean specification, particularly in data-constrained contexts.

It serves as the model applied to the test set in the generalization evaluation below.

## Classification Summary

### Comparative Analysis of Logistic Models

The classification sequence reveals a consistent pattern of diminishing returns as model complexity increases.

**Logit 1 — strong baseline:** Size and quality alone produce a ROC-AUC of approximately 0.96. This result confirms that the binary classification problem is largely solvable with two predictors, and that the majority of classification signal is concentrated in these variables. Any subsequent model must be evaluated against this high baseline.

**Logit 2 — categorical information gain:** Adding simplified neighborhood and kitchen quality yields a modest but genuine ROC-AUC improvement. The primary contribution comes from the neighborhood indicator, which captures location-based price premiums not fully represented by the numeric predictors. The gain confirms that categorical structure adds information even after controlling for size and quality.

**Logit 3 — main jump in classification quality:** The addition of lot area, bathroom count, house age, and renovation recency produces the largest incremental gain in ROC-AUC, reaching approximately 0.975. This model represents the richest specification and demonstrates that the high-price tier is influenced by a broader set of property characteristics beyond the dominant quality-size signal.

**Logit 4 — parsimony without performance loss:** Removing the weakest predictor (garage area) preserves the ROC-AUC of Logit 3 exactly. This is a strong result: it demonstrates that the final performance level is achievable with a model one predictor simpler, and that Logit 3's additional complexity is not justified by its results.

**Logit Final — lean alternative specification:** Removing kitchen quality produces no measurable decline in ROC-AUC. This suggests that the classification problem is robust to the exclusion of this variable once the remaining predictors are in place.

**Trade-off summary:** The classification sequence illustrates a general principle: early additions to a strong baseline produce the largest gains (Logit 3), while later additions produce diminishing returns (Logit 4, Logit Final). Beyond a certain complexity threshold, additional variables contribute instability rather than discrimination. The preferred model is therefore Logit 4 for full-feature contexts and Logit Final for data-constrained contexts.

In [20]:
classification_summary = pd.DataFrame([
    {'Model': 'Logit 1', 'Description': 'Baseline (Qual + SF)',                        'ROC-AUC': roc_auc_score(val_model['is_high'], val_model['pred_prob_1'])},
    {'Model': 'Logit 2', 'Description': '+ Neighborhood + Kitchen Qual',               'ROC-AUC': roc_auc_score(val_model['is_high'], val_model['pred_prob_2'])},
    {'Model': 'Logit 3', 'Description': '+ Extended numerics + categoricals',          'ROC-AUC': roc_auc_score(val_model['is_high'], val_model['pred_prob_3'])},
    {'Model': 'Logit 4', 'Description': 'Parsimonious (removes weak from Logit 3)',   'ROC-AUC': roc_auc_score(val_model['is_high'], val_model['pred_prob_4'])},
    {'Model': 'Logit Final', 'Description': 'Without Kitchen Qual',                   'ROC-AUC': roc_auc_score(val_model['is_high'], val_model['pred_prob_final'])},
])

classification_summary

,Model,Description,ROC-AUC
0,Logit 1,Baseline (Qual + SF),0.960671
1,Logit 2,+ Neighborhood + Kitchen Qual,0.963993
2,Logit 3,+ Extended numerics + categoricals,0.975012
3,Logit 4,Parsimonious (removes weak from Logit 3),0.975060
4,Logit Final,Without Kitchen Qual,0.975084


---
## Post-Hoc Feature Screening

**Audit note:** The following two cells constitute a post-hoc feature screening experiment that evaluates two candidate engineered features (`Qual_Cond`, `Has_Garage`) against a base regression formula. This experiment was part of the model development process and is retained for transparency.

These cells are not part of the main modeling sequence and do not affect the results reported for Models 1–7 or the logistic models. The variable `model` created within the loop shadows no named model object used elsewhere, but should be treated as a transient variable. The screening results informed the selection of `Qual_Cond` for Models 2 and 7.

In [21]:
base_formula = '''
log_SalePrice ~  np.log(Total_SF)
                + np.log(Q("Lot Area"))
                + np.log(Q("Garage Area") + 1)
                + Total_Bath
                + House_Age
                + Years_Since_Remod
                + C(Neighborhood_grouped)
                + C(Q("Bldg Type"))
                + C(Q("Kitchen Qual"))
'''

In [22]:
candidate_features = [
    "Qual_Cond",
    "Has_Garage"
]

results = []

# --- baseline ---
model = smf.ols(base_formula, data=train_model).fit()

preds = model.predict(val_model)

rmse_log = np.sqrt(mean_squared_error(val_model['log_SalePrice'], preds))
rmse_original = np.sqrt(mean_squared_error(
    val_model['SalePrice'], np.exp(preds)
))

results.append({
    "Feature": "BASELINE",
    "RMSE_log": rmse_log,
    "RMSE_original": rmse_original
})


# --- testes ---
for feat in candidate_features:
    formula_test = base_formula + f" + {feat}"

    model = smf.ols(formula_test, data=train_model).fit()

    preds = model.predict(val_model)

    rmse_log = np.sqrt(mean_squared_error(val_model['log_SalePrice'], preds))
    rmse_original = np.sqrt(mean_squared_error(
        val_model['SalePrice'], np.exp(preds)
    ))

    results.append({
        "Feature": feat,
        "RMSE_log": rmse_log,
        "RMSE_original": rmse_original
    })

pd.DataFrame(results).sort_values("RMSE_original")

,Feature,RMSE_log,RMSE_original
1,Qual_Cond,0.155896,23579.116696
2,Has_Garage,0.185523,28448.518826
0,BASELINE,0.185624,28703.798071


---
## Regression — Test Set Generalization

### Model Selection for Test Evaluation

All model comparison and selection decisions described above were made exclusively on the validation set. The test set was withheld throughout the model development process and is used here for the first time to assess generalization performance.

**Model 4 is applied to the test set** as the designated final regression model. It was selected over Model 3 (which had lower validation RMSE) because its simpler categorical structure is more robust to distributional shift in unseen data, and its coefficients are fully interpretable. The selection criterion prioritised parsimony and stability alongside predictive performance.

In [34]:
test_model['pred_log'] = model4.predict(test_model)

# RMSE log
rmse_log_test = np.sqrt(mean_squared_error(
    test_model['log_SalePrice'],
    test_model['pred_log']
))

# RMSE original
test_model['pred_original'] = np.exp(test_model['pred_log'])

rmse_original_test = np.sqrt(mean_squared_error(
    test_model['SalePrice'],
    test_model['pred_original']
))

print(f"Test RMSE (log): {rmse_log_test:.4f}")
print(f"Test RMSE (original): ${rmse_original_test:,.0f}")

Test RMSE (log): 0.1546
Test RMSE (original): $29,476


### Interpretation of Test Results

**Test RMSE (log scale): 0.1546.** The log-scale RMSE on the test set is consistent with the validation RMSE, indicating that the model generalises well to held-out data. The absence of a substantial gap between validation and test performance is evidence that the model is not overfit to the validation set.

**Test RMSE (original scale): $29,476.** On the original dollar scale, the mean prediction error is approximately $29,500. This figure requires careful interpretation: the original-scale RMSE is sensitive to the pricing distribution, and is disproportionately influenced by prediction errors on expensive properties where percentage errors translate to large absolute dollar amounts. A log-scale RMSE of 0.155 corresponds to a geometric mean prediction error of approximately 15–17 percent, which is a more representative summary of model accuracy across the price range.

There is no major evidence of overfitting. The test RMSE is close to the validation RMSE, confirming that the generalisation performance observed during model selection accurately reflects the model's behaviour on unseen data.

---
## Classification — Test Set Generalization

### Model Selection and Threshold

The Logit Final model is applied to the test set. It was selected as the final classification model on the basis of its strong ROC-AUC, lean specification (no kitchen quality predictor), and near-identical performance to Logit 4. The classification threshold is set at 0.4, consistent with all validation evaluations.

**Threshold note:** The default logistic regression threshold is 0.5. A threshold of 0.4 was selected to prioritise recall for the high-price class. In a real estate context, failing to identify an expensive property as high-price (a false negative) is considered a more costly error than incorrectly flagging a mid-price property as high (a false positive). Reducing the threshold from 0.5 to 0.4 increases sensitivity at a modest cost in precision, which is the appropriate trade-off for this task. Any threshold selection introduces a precision–recall trade-off, and this choice should be revisited if the cost structure changes.

In [24]:
test_model['pred_prob'] = model_logit_final.predict(test_model)

In [25]:
threshold = 0.4  # ou o que escolheste

test_model['pred_class'] = (test_model['pred_prob'] >= threshold).astype(int)

In [26]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print(confusion_matrix(test_model['is_high'], test_model['pred_class']))

print(classification_report(test_model['is_high'], test_model['pred_class']))

roc = roc_auc_score(test_model['is_high'], test_model['pred_prob'])
print(f"ROC-AUC: {roc:.4f}")

[[264  22]
 [ 10 144]]
              precision    recall  f1-score   support

           0       0.96      0.92      0.94       286
           1       0.87      0.94      0.90       154

    accuracy                           0.93       440
   macro avg       0.92      0.93      0.92       440
weighted avg       0.93      0.93      0.93       440

ROC-AUC: 0.9825


### Interpretation of Test Results

The confusion matrix `[[264, 22], [10, 144]]` indicates that the model correctly classifies 264 non-high-price properties and 144 high-price properties, with 22 false positives and 10 false negatives.

**Accuracy: 0.93.** The model correctly classifies 93 percent of all test observations. Given the class balance (approximately two-thirds non-high, one-third high), this accuracy is not trivially explained by majority-class prediction and reflects genuine discriminatory power.

**Recall for the high-price class: 0.94.** The model correctly identifies 94 percent of all actual high-price properties. This is the key performance metric given the threshold choice: very few expensive properties are missed. A recall of 0.94 means that only 10 of 154 high-price properties in the test set are misclassified as non-high.

**Precision for the high-price class: 0.87.** Of all properties classified as high-price, 87 percent are genuinely high-price. The 13 percent false positive rate reflects properties near the decision boundary — mid-to-high price properties that share characteristics (location, quality, area) with the high tier.

**F1 for the high-price class: 0.90.** The harmonic mean of precision and recall indicates balanced performance across both error types, weighted equally.

**ROC-AUC: 0.9825.** The test ROC-AUC of 0.9825 exceeds the validation ROC-AUC, confirming that the model generalises extremely well. The classifier exhibits excellent class discrimination: it correctly ranks a randomly selected high-price property above a randomly selected non-high-price property more than 98 percent of the time.

Overall, the Logit Final model is well suited to identifying high-value residential properties. The high recall ensures that expensive homes are rarely missed; the high ROC-AUC confirms that this discriminatory ability is stable and not dependent on the specific threshold chosen.

---
## Final Modeling Conclusions

### Regression

The regression analysis proceeded through seven models of increasing and varying complexity. Several key findings emerge.

The baseline specification (Model 1) was already strong: overall quality, total floor area, and house age collectively explained over 80 percent of log-price variance, establishing that the Ames dataset is well-structured for linear modeling. Feature engineering and log transformation of skewed predictors (Model 2) produced a substantial RMSE improvement, confirming that the form of numeric representation matters independently of variable selection.

The addition of categorical predictors — particularly neighborhood — in Model 3 produced the best validation RMSE among all regression models. Location carries structural pricing information that cannot be proxied by numeric variables. Simplifying the categorical structure in Model 4 accepted a moderate performance cost in exchange for interpretability and stability; Model 4 is designated the final regression model.

Model 5 demonstrated that an interaction between neighborhood and size introduces multicollinearity so severe (VIF > 1,000) that the model is not a viable specification, despite the theoretical interest of the hypothesis. Model 6 confirmed that a minimal four-variable model is insufficient, establishing the lower bound on necessary complexity. Model 7 showed that replacing `Overall Qual` with the composite `Qual_Cond` marginally improves fit but at the cost of losing significance for `Years_Since_Remod`, justifying the retention of `Overall Qual` in the preferred specification.

The test set generalization check (RMSE log: 0.1546, RMSE original: $29,476) is consistent with validation performance and provides no evidence of overfitting.

### Classification

The binary classification of high-price properties was tractable even at the baseline: two predictors (quality and area) achieved a ROC-AUC of approximately 0.96. This confirms that the high-price tier is predominantly determined by the dominant structural attributes, and that the classification problem is well-posed.

Enriching the model with neighborhood, kitchen quality, lot area, bathroom count, house age, and renovation recency (Logit 3) produced the main improvement in discrimination, reaching a ROC-AUC of 0.975. Subsequent parsimony reduction (Logit 4, Logit Final) preserved this performance level, confirming that the full specification is over-parameterised for this task.

The Logit Final model — without kitchen quality — achieves a test ROC-AUC of 0.9825 with accuracy of 0.93 and recall of 0.94 for the high-price class. This confirms strong generalisation: the model's discriminatory ability on held-out data is consistent with or exceeds its validation-set performance.

### Overall

The dominant drivers of sale price — and of high-price tier membership — are property size, overall quality, neighborhood, bathroom count, and renovation recency. These variables jointly capture the structural, locational, and temporal dimensions of residential value. Garage area and kitchen quality, while informative in isolation, contribute only marginal incremental signal once the primary drivers are included.

The modeling process followed a principled structure: iterative complexity expansion on the training set, model selection based on validation-set performance, and final evaluation on a held-out test set that was never used during model development. Model choices were based not only on raw predictive performance but also on coefficient interpretability, multicollinearity diagnostics, and generalization stability. This approach ensures that the selected models are both accurate and reliable outside the training distribution.